# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR<sup>2</sup> dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Here we list all the available record sets, their `@id`, and included fields.

In [ ]:
# List all record sets and their fields with ids
recsets = dataset.metadata.record_sets
if not recsets:
    print('No record sets defined in metadata.')
else:
    for rs in recsets:
        print(f"Record set name: {rs.name}\n  @id: {rs.id}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, data_type: {field.data_type})")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
We demonstrate extracting all records from each available record set.

In [ ]:
# Extract data from each record set
record_sets = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    print(f'Loading records from record set {record_set_id} ...')
    records = list(dataset.records(record_set=record_set_id)) # Records are always dicts keyed by field @id
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Rows: {len(records)} Columns: {dataframes[record_set_id].columns.tolist()}\n")

# For demonstration: show the first record set (if available)
if record_sets:
    first_record_set = record_sets[0]
    print(f"Columns in record set '{first_record_set}':")
    print(dataframes[first_record_set].columns.tolist())
    display(dataframes[first_record_set].head())
else:
    print('No record sets available.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering numeric fields, normalizing values, removing outliers, or grouping data for analysis.

All columns/fields are referenced by their `@id`.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)

# Pick a record set for EDA (use the first if multiple)
if record_sets:
    eda_record_set = record_sets[0]
    df = dataframes[eda_record_set]

    # Identify numeric fields (use Croissant's data_type info)
    record_set_obj = next(rs for rs in dataset.metadata.record_sets if rs.id == eda_record_set)
    numeric_field_ids = [f.id for f in record_set_obj.fields if f.data_type in ['schema:Number', 'schema:Integer', 'schema:Float']]
    print('Numeric fields available:', numeric_field_ids)

    if numeric_field_ids:
        numeric_field = numeric_field_ids[0]  # Choose the first numeric field

        # Try to convert the field to numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = np.nanmean(df[numeric_field]) if df[numeric_field].notnull().any() else 0
        print(f'Using threshold: {threshold} for field {numeric_field}\n')

        # Filter records
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        mean = filtered_df[numeric_field].mean()
        std = filtered_df[numeric_field].std()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field, if available
        # Identify non-numeric fields
        non_numeric_field_ids = [f.id for f in record_set_obj.fields if f.data_type not in ['schema:Number', 'schema:Integer', 'schema:Float', 'schema:Boolean']]
        if non_numeric_field_ids:
            group_field = non_numeric_field_ids[0]
            print(f'Grouping by {group_field}...')
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(f'Mean {numeric_field}')
                print(f"\nGrouped mean of {numeric_field} by {group_field}:")
                display(grouped_df.head())
            else:
                print(f"Group field {group_field} is not present in DataFrame columns.")
        else:
            print("No categorical fields available for grouping.")
    else:
        print('No numeric fields present in this record set.')
else:
    print('No record set available for EDA.')

## 5. Visualization
Visualize numeric field distributions or relationships between fields.

We'll plot a histogram for the numeric field and a boxplot per group (if grouping fields were available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and 'numeric_field' in locals():
    df = dataframes[eda_record_set]

    # Drop NA
    numeric_values = df[numeric_field].dropna().values
    plt.figure(figsize=(8,4))
    plt.hist(numeric_values, bins=30, color='skyblue', edgecolor='black')
    plt.title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(12,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f'Boxplot of {numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric fields available for visualization.')

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the [FAIR<sup>2</sup> dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. We examined available record sets and fields via their `@id`, extracted their records, and performed basic EDA and visualization using dynamically discovered field identifiers. This approach can be adapted to other datasets defined with Croissant schemas.